## Step 2: Tamper-Evident Blockchain Trust Ledger (SHA-256 Hash-Chain)

A hash chain reveals edits **relative to a trusted checkpoint**. Whoever controls the whole chain can edit a block and recompute every later hash; the chain then still verifies internally. Detecting that requires an *externally anchored* head hash (`head_hash()` / `is_valid(anchor_hash=...)`). The self-test in the next cell demonstrates both cases.


In [ ]:
class TrustBlock:
    """
    A single block in the trust ledger.
    Each block stores per-client trust scores for one FL round and
    is cryptographically linked to the previous block via SHA-256.
    Editing one block breaks its own hash and every later link, UNLESS the editor
    recomputes the whole chain — which is only detectable against an externally
    anchored head hash (see TrustBlockchain.is_valid(anchor_hash=...)).
    """
    def __init__(self, index, round_num, trust_scores, crtat_scores,
                 flagged_clients, prev_hash):
        self.index           = index
        self.round_num       = round_num
        self.timestamp       = time.time()
        self.trust_scores    = trust_scores          # dict {cid: float}
        self.crtat_scores    = crtat_scores          # dict {cid: float}
        self.flagged_clients = flagged_clients       # list of flagged cids
        self.prev_hash       = prev_hash
        self.hash            = self._compute_hash()

    def _compute_hash(self):
        block_str = _json.dumps({
            'index'          : self.index,
            'round_num'      : self.round_num,
            'timestamp'      : self.timestamp,
            'trust_scores'   : {str(k): round(float(v),8)
                                for k,v in self.trust_scores.items()},
            'crtat_scores'   : {str(k): round(float(v),8)
                                for k,v in self.crtat_scores.items()},
            'flagged_clients': self.flagged_clients,
            'prev_hash'      : self.prev_hash,
        }, sort_keys=True)
        return hashlib.sha256(block_str.encode()).hexdigest()

    def __repr__(self):
        return (f"Block(round={self.round_num}, "
                f"flagged={self.flagged_clients}, "
                f"hash={self.hash[:12]}...)")


class TrustBlockchain:
    """
    Append-only blockchain that stores trust audit records for each FL round.
    Provides:
      • add_round()     — append a new trust record block
      • is_valid()      — verify chain integrity (tamper detection)
      • get_history()   — retrieve full trust history per client
      • audit_report()  — print human-readable ledger summary
    """
    GENESIS_HASH = '0' * 64

    def __init__(self):
        self.chain  = []
        self._create_genesis()

    def _create_genesis(self):
        genesis = TrustBlock(
            index=0, round_num=0,
            trust_scores={}, crtat_scores={},
            flagged_clients=[],
            prev_hash=self.GENESIS_HASH
        )
        self.chain.append(genesis)

    def add_round(self, round_num, trust_scores, crtat_scores,
                  flagged_clients, threshold=0.6):
        """Append a new block for this FL round."""
        prev_hash = self.chain[-1].hash
        block = TrustBlock(
            index           = len(self.chain),
            round_num       = round_num,
            trust_scores    = {i: float(v) for i,v in enumerate(trust_scores)},
            crtat_scores    = {i: float(v) for i,v in enumerate(crtat_scores)},
            flagged_clients = [int(i) for i,v in enumerate(crtat_scores)
                               if v >= threshold],
            prev_hash       = prev_hash
        )
        self.chain.append(block)
        return block

    def head_hash(self):
        """Hash of the latest block. Publish/store it OUTSIDE this ledger to anchor it."""
        return self.chain[-1].hash

    def is_valid(self, anchor_hash=None):
        """Verify internal consistency (each hash matches its content; links correct).
        anchor_hash: optional head hash recorded in a different trust domain. Without it,
        a party that controls the entire chain can rewrite a block, recompute all later
        hashes, and is_valid() will still pass."""
        for i in range(1, len(self.chain)):
            curr, prev = self.chain[i], self.chain[i-1]
            if curr.hash != curr._compute_hash():
                return False, f'Block {i} hash mismatch — tampered!'
            if curr.prev_hash != prev.hash:
                return False, f'Block {i} broken link — chain compromised!'
        if anchor_hash is not None and self.chain[-1].hash != anchor_hash:
            return False, 'Head hash differs from the external anchor — history rewritten!'
        return True, 'Chain valid ✅'

    def get_history(self, cid):
        """Return list of (round, trust, crtat) for a given client."""
        history = []
        for block in self.chain[1:]:
            t = block.trust_scores.get(cid, None)
            c = block.crtat_scores.get(cid, None)
            if t is not None:
                history.append((block.round_num, t, c))
        return history

    def cumulative_flags(self, cid):
        """How many rounds was this client flagged?"""
        return sum(1 for b in self.chain[1:] if cid in b.flagged_clients)

    def audit_report(self, n_clients, malicious_ids, last_n=5):
        """Print a summary of the last n rounds from the ledger."""
        print(f'\n{"═"*72}')
        print(f'  BLOCKCHAIN TRUST LEDGER AUDIT — Last {last_n} Rounds')
        print(f'  Chain length : {len(self.chain)} blocks')
        valid, msg = self.is_valid()
        print(f'  Integrity    : {msg}')
        print(f'{"═"*72}')
        for block in self.chain[-last_n:]:
            if block.index == 0: continue
            flagged = block.flagged_clients
            print(f'  Block #{block.index:<3} | Round {block.round_num:<3} | '
                  f'Hash: {block.hash[:16]}... | '
                  f'Flagged: {flagged}')
        print(f'{"═"*72}')
        print(f'  Cumulative flag counts (all rounds):')
        for cid in range(n_clients):
            flags = self.cumulative_flags(cid)
            lbl   = '⚠️ MALICIOUS' if cid in malicious_ids else '  honest    '
            bar   = '█' * flags + '░' * (len(self.chain)-1-flags)
            print(f'    C{cid:02d} [{lbl}] {bar} ({flags} flags)')
        print(f'{"═"*72}\n')

print('✅ TrustBlockchain defined.')
print('   SHA-256 hash-chain: each block stores per-client trust + CRTAT scores.')
print('   Tamper-EVIDENT relative to a trusted/anchored checkpoint (not tamper-proof).')

# Quick self-test
_bc = TrustBlockchain()
_bc.add_round(1, np.random.rand(5), np.random.rand(5), [0,2])
_bc.add_round(2, np.random.rand(5), np.random.rand(5), [0])
valid, msg = _bc.is_valid()
print(f'   Self-test: {msg}')
del _bc

# What the hash chain does and does NOT guarantee
_bc = TrustBlockchain()
for _r in range(1, 4): _bc.add_round(_r, np.random.rand(5), np.random.rand(5), [])
_anchor = _bc.head_hash()                                   # stored outside the ledger
_bc.chain[2].trust_scores[0] = 0.0                          # (1) naive edit of one block
print('   (1) naive edit              →', _bc.is_valid()[1])
for _i in range(2, len(_bc.chain)):                         # (2) edit + recompute all later hashes
    if _i > 2: _bc.chain[_i].prev_hash = _bc.chain[_i-1].hash
    _bc.chain[_i].hash = _bc.chain[_i]._compute_hash()
print('   (2) edit + rehash, no anchor →', _bc.is_valid()[1], ' ← internally consistent!')
print('   (2) edit + rehash, anchored  →', _bc.is_valid(anchor_hash=_anchor)[1])
del _bc


✅ TrustBlockchain defined.
   SHA-256 hash-chain: each block stores per-client trust + CRTAT scores.
   Tamper-EVIDENT relative to a trusted/anchored checkpoint (not tamper-proof).
   Self-test: Chain valid ✅
   (1) naive edit              → Block 2 hash mismatch — tampered!
   (2) edit + rehash, no anchor → Chain valid ✅  ← internally consistent!
   (2) edit + rehash, anchored  → Head hash differs from the external anchor — history rewritten!
